# Water quality data access


In [6]:
from pathlib import Path
import requests
import pandas as pd
import time



All downloaded data are stored **outside the repository** to:

- keep the repo lightweight. Note: never store files more than 50 MB in your research repository, this will violate GitHub use policies.
- prevent large file versioning.
- maintain reproducibility.
- separate raw vs processed data.

The location of raw data is determined automatically using the project name.

In [12]:
from pathlib import Path

notebook_dir = Path.cwd()
home_dir = Path.home()
raw_data_dir = home_dir / "WaterDig/Project3"
raw_data_dir.mkdir(parents=True, exist_ok=True)
print("External raw data directory:")
print(raw_data_dir.resolve())

External raw data directory:
/home/jovyan/WaterDig/Project3


### Make sure you have sufficient storage available to download all data required

## API query used

Base endpoint:

https://rajapinnat.ymparisto.fi/api/vesla/2.0/odata/Result_Wide

Paginated requests:

https://rajapinnat.ymparisto.fi/api/vesla/2.0/odata/Result_Wide?$skip=1000

https://rajapinnat.ymparisto.fi/api/vesla/2.0/odata/Result_Wide?$skip=2000

...

https://rajapinnat.ymparisto.fi/api/vesla/2.0/odata/Result_Wide?$skip=10000


Filtering applied in Python:

Determination_Id == 1

In [13]:
import requests
import pandas as pd
import time

# =========================
# USER SETTINGS
# =========================
BASE_URL = "https://rajapinnat.ymparisto.fi/api/vesla/2.0/odata/Result_Wide?$skip={}"
OUTPUT_FILE = raw_data_dir / "aci_records.csv"

INITIAL_SKIP = 1000
MAX_SKIP = 10000
STEP = 1000
TARGET_DETERMINATION_ID = 1
DELAY_SECONDS = 0.2
TIMEOUT = 60
# =========================

all_records = []

print("Starting API data retrieval...")
print(f"Target Determination_Id: {TARGET_DETERMINATION_ID}")
print(f"Output file: {OUTPUT_FILE}")

for skip_value in range(INITIAL_SKIP, MAX_SKIP + 1, STEP):
    url = BASE_URL.format(skip_value)
    print(f"Fetching skip={skip_value}")

    try:
        response = requests.get(url, timeout=TIMEOUT)
        response.raise_for_status()

        data = response.json()
        records = data.get("value", [])

        if not records:
            print(f"No records returned at skip={skip_value}. Stopping.")
            break

        filtered_records = [
            record for record in records
            if record.get("Determination_Id") == TARGET_DETERMINATION_ID
        ]

        all_records.extend(filtered_records)

        print(f"  Retrieved {len(records)} records, matched {len(filtered_records)} records.")

        # If fewer than STEP records are returned, we may have reached the end
        if len(records) < STEP:
            print("Fewer records than page size returned. Assuming end of dataset.")
            break

        time.sleep(DELAY_SECONDS)

    except requests.exceptions.RequestException as e:
        print(f"Request failed at skip={skip_value}: {e}")
        break
    except ValueError as e:
        print(f"JSON parsing failed at skip={skip_value}: {e}")
        break

df_aci = pd.DataFrame(all_records)

if not df_aci.empty:
    df_aci.to_csv(OUTPUT_FILE, index=False)
    print(f"\nSaved {len(df_aci)} filtered records to:")
    print(OUTPUT_FILE)
else:
    print("\nNo matching records found.")

Starting API data retrieval...
Target Determination_Id: 1
Output file: /home/jovyan/WaterDig/Project3/aci_records.csv
Fetching skip=1000
  Retrieved 1000 records, matched 1000 records.
Fetching skip=2000
  Retrieved 1000 records, matched 1000 records.
Fetching skip=3000
  Retrieved 1000 records, matched 1000 records.
Fetching skip=4000
  Retrieved 1000 records, matched 1000 records.
Fetching skip=5000
  Retrieved 1000 records, matched 1000 records.
Fetching skip=6000
  Retrieved 1000 records, matched 1000 records.
Fetching skip=7000
  Retrieved 1000 records, matched 1000 records.
Fetching skip=8000
  Retrieved 1000 records, matched 1000 records.
Fetching skip=9000
  Retrieved 1000 records, matched 1000 records.
Fetching skip=10000
  Retrieved 1000 records, matched 1000 records.

Saved 10000 filtered records to:
/home/jovyan/WaterDig/Project3/aci_records.csv


In [14]:
print("Files in raw_data_dir:")
print(list(raw_data_dir.glob("*")))

print("\nPreview of retrieved data:")
print(df_aci.head())
print("\nShape:", df_aci.shape)

Files in raw_data_dir:
[PosixPath('/home/jovyan/WaterDig/Project3/.ipynb_checkpoints'), PosixPath('/home/jovyan/WaterDig/Project3/aci_records.csv'), PosixPath('/home/jovyan/WaterDig/Project3/Untitled.ipynb')]

Preview of retrieved data:
   Site_Id                   Site  Sampling_id                       Time  \
0     2885  Ruuhijärvi eteläosa 1        29650  1986-07-17T00:00:00+02:00   
1     2885  Ruuhijärvi eteläosa 1        29651  1987-08-25T00:00:00+02:00   
2     2885  Ruuhijärvi eteläosa 1        29651  1987-08-25T00:00:00+02:00   
3     2885  Ruuhijärvi eteläosa 1        29651  1987-08-25T00:00:00+02:00   
4     2885  Ruuhijärvi eteläosa 1        29651  1987-08-25T00:00:00+02:00   

  SampleDepth_m SampleDepthUpper_m SampleDepthLower_m  Sample_Id  \
0          17.0               17.0                NaN      74190   
1           1.0                1.0                NaN      74191   
2           5.0                5.0                NaN      74192   
3          10.0             

In [11]:
import shutil

total, used, free = shutil.disk_usage(Path.home())
print("Total GB:", round(total / 1024**3, 2))
print("Used GB:", round(used / 1024**3, 2))
print("Free GB:", round(free / 1024**3, 2))

Total GB: 491.08
Used GB: 14.4
Free GB: 476.66



**This notebook accesses raw water-quality data through the VESLA OData API.  
The API is queried using paginated requests with the `$skip` parameter.  
The raw JSON responses are filtered to retain only records with `Determination_Id == 1`, then saved as a CSV file in the raw data directory outside the repository.

This approach supports reproducibility because:
- the source is an online API,
- the full query structure is shown,
- output is saved in a stable raw-data folder outside the git repository,
- no manual downloading is required.**